# Temperature-mortality response and humidity modification

This notebook produces **Manuscript Figure 2**. It evaluates the fitted weather response using a common standard age distribution so model configurations can be compared independently of Germany's observed age structure.

The left panel compares the hybrid model with a weather-only model. The right panel compares air-temperature-only estimates with air temperature plus dew point under fixed relative-humidity levels. Each synthetic exposure is held constant across the full six-day weather input window. Curves show additional mortality risk relative to each model's minimum-risk temperature.

## Execution mode

Set `EXECUTION_DEVICE` in the first code cell to `"auto"`, `"cpu"`, or `"cuda"`. Auto mode uses CUDA when available and CPU otherwise. CPU mode evaluates only the `v0` checkpoint for a faster exploratory run; CUDA mode evaluates the complete 10-checkpoint ensemble used for manuscript results.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'model').is_dir() and (p / 'notebooks').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate the repository root')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from model.containers import MortalityModelInputs
from notebooks import dew_point_from_relative_humidity, load_checkpoint, load_experiment, resolve_device

EXECUTION_DEVICE = "auto"
DEVICE = resolve_device(EXECUTION_DEVICE)

In [2]:
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)
mpl.rcParams.update({
    "font.family": "serif",
    "font.sans-serif": ["Arial"],
    "mathtext.fontset": "cm",
    "axes.labelsize": 12,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "axes.titleweight": "bold",
    "axes.titlesize": 13,
    "lines.linewidth": 1.2,
    "grid.alpha": 0.4,
})

## Hybrid versus weather-only response

The first comparison evaluates models with the same temperature and dew-point inputs but with or without respiratory-infection indicators. Mortality is calculated against a shared standard age distribution, and uncertainty is represented by the range across selected checkpoints.

In [3]:
# Comparing models with and without SARI features. Illustrating the effect of SARI features on the learned temperature-mortality relationship.

model = dict()
res = dict()
std_age = torch.tensor([88569, 86870, 85970, 84670, 82171, 79272, 76073, 71475, 65877, 60379, 53681, 45484, 37187, 29590, 22092, 15195, 9097, 4398, 1500, 400, 50]).float()
std_age = torch.cat([std_age[:6].sum(0, keepdim=True), std_age[6:-2], std_age[-2:].sum(0, keepdim=True)]).unsqueeze(-1)
setup = ['HeatMortality_EXP_t_6_sari_10_covid_median_ewma_corr_on_pop_average_tas-tdew_stage1', 
         'HeatMortality_EXP_t_6_median_ewma_corr_on_pop_average_tas-tdew_stage1']
for v in setup:
    model[v], checkpoint_paths = load_experiment(v, DEVICE)
    res[v] = []
    with torch.no_grad():
        for checkpoint_path in checkpoint_paths:
            load_checkpoint(model[v], checkpoint_path, DEVICE)
            model[v].evaluate_baseline()
            risk = []
            for i in np.arange(-5, 28.5, 0.5):
                temp = torch.tensor([i, dew_point_from_relative_humidity(i, 0.5)]).unsqueeze(0).unsqueeze(2).repeat(1,1,6)
                outputs = model[v](MortalityModelInputs(temp.float().to(DEVICE), 
                                    torch.zeros([15,10], dtype=torch.float, device=DEVICE),
                                    torch.zeros([15,10], dtype=torch.float, device=DEVICE)))
                t_factors = outputs.temperature_effect.abs()
                mort = model[v].basic_death_rate_range[-1] * std_age.to(DEVICE) * t_factors
                mort = model[v].apply_training_mean_correction_tensor(mort).sum().cpu()
                risk.append(mort)
            res[v].append(risk)

risk = dict()
for v in setup:
    risk[v] = torch.stack([torch.stack(r) for r in res[v]]).detach().cpu()
        

HeatMortality_EXP_t_6_sari_10_covid_median_ewma_corr_on_pop_average_tas-tdew_stage1: device=cpu, checkpoints=1 (v0)
HeatMortality_EXP_t_6_median_ewma_corr_on_pop_average_tas-tdew_stage1: device=cpu, checkpoints=1 (v0)


## Air-temperature-only response

This model omits dew point and provides the reference curve used to assess how an explicit humidity indicator changes the fitted heat response.

In [ ]:
# Using only air temperature without humidity.
res_1 = []

v = 'HeatMortality_EXP_t_6_sari_10_covid_median_ewma_corr_on_pop_average_tas_stage1'

model_1, checkpoint_paths = load_experiment(v, DEVICE)
for checkpoint_path in checkpoint_paths:
    load_checkpoint(model_1, checkpoint_path, DEVICE)
    with torch.no_grad():
        model_1.evaluate_baseline()
        risk_temp = []
        for i in np.arange(-5, 32.5, 0.5):
            temp = torch.tensor([i]).unsqueeze(0).unsqueeze(2).repeat(1,1,6)      
            outputs = model_1(MortalityModelInputs(
                temp.float().to(DEVICE),
                torch.zeros([15,10], dtype=torch.float, device=DEVICE),
                torch.zeros([15,10], dtype=torch.float, device=DEVICE),
            ))
            t_factors = outputs.temperature_effect.abs().detach()
            mort = model_1.basic_death_rate_range[-1] * std_age.to(DEVICE) * t_factors
            mort = model_1.apply_training_mean_correction_tensor(mort).sum().cpu()
            risk_temp.append(mort)
        res_1.append(risk_temp)

risk_1 = torch.stack([torch.stack(r) for r in res_1]).detach().cpu()

## Fixed relative-humidity scenarios

The air-temperature-plus-dew-point model is evaluated from 30% to 80% relative humidity. Dew point is derived from each air-temperature and relative-humidity pair before the six-day synthetic input is passed to the model.

In [ ]:
# Estimate risk for different humidity levels.
res_2 = dict()

v = 'HeatMortality_EXP_t_6_sari_10_covid_median_ewma_corr_on_pop_average_tas-tdew_stage1'

model_2, checkpoint_paths = load_experiment(v, DEVICE)
for checkpoint_path in checkpoint_paths:
    load_checkpoint(model_2, checkpoint_path, DEVICE)
    model_2.evaluate_baseline()

    with torch.no_grad():
        for h in range(3, 9, 1):
            if h not in res_2.keys():
                res_2[h] = []
            risk_temp = []
            for i in np.arange(-5, 32.5, 0.5):       
                temp = torch.tensor([i, dew_point_from_relative_humidity(i, h/10)]).unsqueeze(0).unsqueeze(2).repeat(1,1,6)    
                outputs = model_2(MortalityModelInputs(
                    temp.float().to(DEVICE),
                    torch.zeros([15,10], dtype=torch.float, device=DEVICE),
                    torch.zeros([15,10], dtype=torch.float, device=DEVICE),
                ))
                t_factors = outputs.temperature_effect.abs()
                mort = model_2.basic_death_rate_range[-1] * std_age.to(DEVICE) * t_factors
                mort = model_2.apply_training_mean_correction_tensor(mort).sum().cpu()
                risk_temp.append(mort)
            res_2[h].append(risk_temp)


risk_2 = dict()
for h in res_2.keys():
    risk_2[h] = torch.stack([torch.stack(r) for r in res_2[h]]).detach().cpu()  


## Manuscript Figure 2: Response curves

Risk is expressed as `(RR - 1) × 100` relative to the minimum of each fitted curve. Dashed grey segments identify temperature-humidity combinations outside the observed humidity range and should not be interpreted as empirically supported exposure combinations.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np


# Reset then set serif font
mpl.rcdefaults()
mpl.rcParams['font.family'] = 'DejaVu Serif'

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax1, ax2 = axes

# ===== Left plot: Temperature–mortality curve =====
x = np.arange(-5, 28.5, 0.5)
for k, label in zip(setup, ['Hybrid Model', 'Air Temperature + Dew Point']):
    # Central line: unchanged
    y = (risk[k].mean(0) / risk[k].mean(0).min() - 1) * 100

    # Range: normalize each curve first, then take min/max over first dimension
    r = (risk[k] / risk[k].min(dim=1, keepdims=True).values - 1) * 100
    lo = r.min(axis=0).values
    hi = r.max(axis=0).values

    line, = ax1.plot(
        x,
        y,
        label=label,
        linewidth=1.5
    )

    ax1.fill_between(
        x,
        lo,
        hi,
        alpha=0.2,
        color=line.get_color(),
        linewidth=0
    )

ax1.set_xlabel('Temperature (°C)')
ax1.set_ylabel('Estimated additional mortality risk\n(relative to minimum-risk temperature)')
# ax1.set_title('Temperature–Mortality Relationship', weight='bold')
ax1.legend(loc='best')
ax1.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
ax1.set_xticks(np.arange(-10, 35, 5))

# ===== Right plot: Bar comparison =====
humidity_temp_map = {
    30: (1.9, 31.6),
    40: (-4.0, 32.5),
    50: (-7.8, 31.7),
    60: (-11.7, 30.2),
    70: (-13.0, 29.0),
    80: (-14.2, 26.6),
    90: (-14.3, 23.8),
    100: (-12.4, 21.3),
}

x = np.arange(-5, 32.5, 0.5)

# Reference: air temperature only
y_ref = (risk_1.mean(0) / risk_1.mean(0).min() - 1) * 100
ax2.plot(
    x,
    y_ref,
    label='Air Temperature only',
    linewidth=2,
    color='black'
 )

# Humidity-specific curves: solid in valid range, dashed grey outside
for h in sorted(risk_2.keys()):
    rh = h * 10
    y = (risk_2[h].mean(0) / risk_2[h].mean(0).min() - 1) * 100
    t_min, t_max = humidity_temp_map.get(rh, (x.min(), x.max()))

    mask_in = (x >= t_min) & (x <= t_max)
    mask_left = x < t_min
    mask_right = x > t_max

    idx_in = np.where(mask_in)[0]
    if idx_in.size == 0:
        continue

    # Plot valid interval in normal color
    ax2.plot(
        x[idx_in],
        y[idx_in],
        label=f'Air Temperature + RH={rh}%',
        linewidth=2,
        zorder=3
    )

    # Plot out-of-range intervals in grey dashed style and bridge with boundary points
    if np.any(mask_left):
        idx_left = np.where(mask_left)[0]
        idx_left_bridge = np.concatenate([idx_left, [idx_in[0]]])
        ax2.plot(
            x[idx_left_bridge],
            y[idx_left_bridge],
            color='0.6',
            linestyle='--',
            linewidth=1.5,
            zorder=2
        )

    if np.any(mask_right):
        idx_right = np.where(mask_right)[0]
        idx_right_bridge = np.concatenate([[idx_in[-1]], idx_right])
        ax2.plot(
            x[idx_right_bridge],
            y[idx_right_bridge],
            color='0.6',
            linestyle='--',
            linewidth=1.5,
            zorder=2
        )

# Add one legend entry for dashed grey segments
ax2.plot([], [], color='0.6', linestyle='--', linewidth=1.5, label='Outside humidity range')

ax2.set_xlabel('Temperature (°C)')
ax2.set_ylabel('Estimated additional mortality risk\n(relative to minimum-risk temperature)')
ax2.set_ylim(0, 800)

ax2.legend(loc='best')
ax2.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
ax2.set_xticks(np.arange(-10, 35, 5))

from matplotlib.ticker import FuncFormatter
percent_fmt = FuncFormatter(lambda y, _: f'{y:.0f}%')

ax1.yaxis.set_major_formatter(percent_fmt)
ax2.yaxis.set_major_formatter(percent_fmt)

# ===== Layout =====
plt.tight_layout()
# plt.savefig('combined_plots.png', dpi=300, bbox_inches='tight')
plt.show()
